In [1]:
import torch
import pickle
import numpy as np
import pandas as pd
import os
import json 
from os.path import dirname



root_path = dirname(os.getcwd()) + "/AdaTest"

pd.set_option("display.max_columns", None)
# data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/comuzzi/_processed/"
data_dir_graphs = root_path + "/data/datasets/comuzzi/graphs_repair/"

print(root_path, data_dir_processed, data_dir_graphs, sep="\n")

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = "cpu"

/home/sebastiano.dissegna/AdaTest
/home/sebastiano.dissegna/AdaTest/data/datasets/comuzzi/_processed/
/home/sebastiano.dissegna/AdaTest/data/datasets/comuzzi/graphs_repair/


In [ ]:
#ACT_TIME_ONLY = True

In [2]:
ACT_TIME_ONLY = False

In [3]:
with open("data/dataset_features.json", 'r') as file:
    datasets_info = json.load(file)

In [4]:
list(datasets_info.keys())

['BPI_Challenge_2013_open_problems',
 'sp2020',
 'Helpdesk',
 'BPI20_RequestForPayment',
 'BPI Challenge 2017 - Offer log',
 'BPI_Challenge_2012_W_Complete',
 'BPI_Challenge_2012_A',
 'bpi_2012_CZ',
 'bpi_2013_CZ',
 'large_log_CZ',
 'small_log_CZ',
 'sp2020_CZ',
 'BPI20_RequestForPayment_CZ']

In [5]:
dataset = "BPI20_RequestForPayment_CZ"

In [6]:
with open("data/dataset_features.json", 'r') as file:
    dataset_info = json.load(file)[dataset]

In [7]:
tab_all = pd.read_csv(f"{data_dir_processed}/{dataset}/{dataset}_processed_all.csv") 
tab_all.head()

,CaseID,Activity,time:timestamp,org:resource,org:role,case:Project,case:Task,case:OrganizationalEntity,case:Cost Type,case:RequestedAmount,case:Activity,case:RfpNumber
0,1,Request For Payment SUBMITTED by EMPLOYEE,0.000000,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
1,1,Request For Payment FINAL_APPROVED by SUPERVISOR,3.761200,STAFF MEMBER,SUPERVISOR,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
2,1,Request For Payment REJECTED by MISSING,11.499992,STAFF MEMBER,MISSING,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
3,1,Request For Payment SUBMITTED by EMPLOYEE,15.337479,STAFF MEMBER,EMPLOYEE,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215
4,1,Request For Payment APPROVED by PRE_APPROVER,15.337486,STAFF MEMBER,PRE_APPROVER,project 148216,UNKNOWN,organizational unit 65463,0,34.336343,UNKNOWN,request for payment number 148215


In [8]:
print(f"number of events {len(tab_all)}")
print(f"number of traces {len(tab_all['CaseID'].unique())}")
print(f"number of attributes {len(tab_all.columns) - 1}") 

number of events 36796
number of traces 6886
number of attributes 11


In [9]:
MISSING_VALUE = "MISSING_VALUE"

In [10]:
if ACT_TIME_ONLY:
    categorical_columns = ["Activity"]
    real_value_columns = ["time:timestamp"]
    dataset = f"{dataset}_AT_only"
else:
    categorical_columns = dataset_info["categorical"]
    real_value_columns = dataset_info["numerical"]

In [11]:
list_unique = {k : list(tab_all[k].unique()) + [MISSING_VALUE] for k in categorical_columns}
#list_unique

In [12]:
import random

torch.manual_seed(0)
torch.cuda.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [13]:
with open(data_dir_graphs + dataset + "_TRAIN_V2_repair.pkl", "rb") as f:
    X_train = pickle.load(f)
with open(data_dir_graphs + dataset + "_VALID_V2_repair.pkl", "rb") as f:
    X_valid = pickle.load(f)
with open(data_dir_graphs + dataset + "_TEST_V2_repair.pkl", "rb") as f:
    X_test = pickle.load(f)

In [14]:
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToUndirected

transform = ToUndirected()


with torch.no_grad():
        for i in range(len(X_train)):
                X_train[i] = transform(X_train[i])
        for i in range(len(X_valid)):
                X_valid[i] = transform(X_valid[i])
        for i in range(len(X_test)):
                X_test[i] = transform(X_test[i])

In [15]:
edge_types = set()
node_types = set()
for i in range(len(X_train)):
    n, edge_type = X_train[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_valid)):
    n, edge_type = X_valid[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
for i in range(len(X_test)):
    n, edge_type = X_test[i].metadata()
    for x in n:
        node_types.add(x)
    for x in edge_type:
        edge_types.add(x)
        
node_types = list(node_types)
edge_types = list(edge_types)

## Hyperopt

In [16]:
from ax.service.managed_loop import optimize
from torch_geometric.nn import (
    HeteroConv,
    SAGEConv
)
from torch.nn import (
    ModuleList,
    Module,
    Linear,
    ModuleDict
  )
from typing_extensions import Self

In [20]:
class HGNN(Module):

    def __init__(self, output_cat, output_real,nodes_relations, parameters) -> Self:  # type: ignore
        super().__init__()

        # List of convolutional layers
        
        hid = parameters["hid"]
        layers = parameters["layers"]
        aggregation = parameters["aggregation"]
       
        
        self.output_cat = output_cat
        self.output_real = output_real
        
        self.convs = ModuleList()
        for _ in range(layers):
            conv = HeteroConv(
                {
                    relation: (
                        #TransformerConv((-1,-1), hid)
                        SAGEConv((-1,-1), aggr=aggregation, out_channels=hid, normalize=False)
                        # GATv2Conv((-1,-1), add_self_loops=False, out_channels=hid, concat=False)
                    )
                    for relation in nodes_relations
                },
                aggr=aggregation,
            )

            self.convs.append(conv)

        self.FC = ModuleDict()
        
        for k in output_cat:
            self.FC[k] = Linear(hid, output_cat[k], device=device)
        for k in output_real:
            self.FC[k] = Linear(hid, 1, device=device)
        
    
        
        

    def forward(self, batch):

        for i in range(len(self.convs)):
            batch.x_dict = self.convs[i]( 
                batch.x_dict, batch.edge_index_dict
            )
            batch.x_dict = {key: x.relu() for key, x in batch.x_dict.items()}

        output = {}
        
        for k in self.output_cat:
            output[k] = self.FC[k](batch.x_dict[k][batch.masks[k]])
        for k in self.output_real:
            output[k] = self.FC[k](batch.x_dict[k][batch.masks[k]]).reshape(1,-1)[0]
            

        return output

In [21]:
from torcheval.metrics.functional import multiclass_accuracy, multiclass_f1_score
import torch.nn as nn
from copy import deepcopy
from tqdm.notebook import tqdm


def train_hgnn(config, output_cat, output_real, epochs=20):
    print(config)

    net = HGNN(
        parameters=config,
        output_cat=output_cat,
        output_real=output_real,
        nodes_relations=edge_types,
    )
    net = net.to(device)

    losses = {}

    for k in output_cat:
        losses[k] = nn.CrossEntropyLoss()
    for k in output_real:
        losses[k] = nn.L1Loss()

    train_loader = DataLoader(X_train, batch_size=config["batch_size"], shuffle=True)
    valid_loader = DataLoader(X_valid, batch_size=config["batch_size"], shuffle=True)

    optimizer = torch.optim.Adam(net.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    best_model = None
    best_loss = 0
    patience = 5
    pat_count = 0

    torch.cuda.empty_cache()

    for epoch in tqdm(range(0, epochs)):

        net.train()
        for _, x in enumerate(train_loader):
            x = x.to(device)

            # TESTTTTTTTTTTT
            #x.mask = torch.full_like(x.mask, fill_value=True)
            #########################
            
            all_labels = x.y
            labels = {k: all_labels[k][x.masks[k]] for k in all_labels}

            optimizer.zero_grad()
            outputs = net(x)

            losses_step = {k: losses[k](outputs[k], labels[k]) for k in losses}

            total_loss = 0
            for k in losses_step:
                total_loss += losses_step[k]

            total_loss.backward()
            optimizer.step()

        running_total_loss = []

        net.eval()
        with torch.no_grad():
            for i, x in enumerate(valid_loader):
                x = x.to(device)

                all_labels = x.y
                labels = {k: all_labels[k][x.masks[k]] for k in all_labels}

                outputs = net(x)

                losses_step = {k: losses[k](outputs[k], labels[k]) for k in losses}

                running_total_loss.append(sum(list(losses_step.values())))

        val_loss = sum(running_total_loss) / len(running_total_loss)

        if epoch == 0:
            best_model = deepcopy(net)
            best_loss = val_loss
        else:
            if val_loss < best_loss:
                best_loss = val_loss
                best_model = deepcopy(net)
                pat_count = 0
            if pat_count == patience:
                return best_model
        pat_count += 1

    return best_model

In [22]:
from torch.nn.functional import l1_loss
 
def test_hgnn(net, output_cat, output_real, test_graphs=X_test):
    test_loader = DataLoader(test_graphs, batch_size=128, shuffle=False)
    
    losses = {}
    
    for k in output_cat:
        losses[k] = (
            nn.CrossEntropyLoss()
        )
    for k in output_real:
        losses[k] = nn.L1Loss()
    
    
    
    
    
    predictions_categorical = {k: [] for k in output_cat}
    target_categorical = {k: [] for k in output_cat}

    avg_MAE = {k : [] for k in output_real}
    prediction_numerical = {k: [] for k in output_real}
    target_numerical = {k: [] for k in output_real}
    
    total_loss = []
        
    net.eval()
    with torch.no_grad():
        for _, x in enumerate(test_loader):
            x = x.to(device)
            
            all_labels = x.y
            labels = {k : all_labels[k][x.masks[k]] for k in all_labels}
            
            outputs = net(x)
            
     
            losses_step = {k: losses[k](outputs[k], labels[k]).item() for k in losses}
            total_loss.append(sum(list(losses_step.values())))
            
            for k in output_cat:
                predictions_categorical[k].append(
                        torch.argmax(torch.softmax(outputs[k], dim=1), 1)
                )
                target_categorical[k].append(labels[k])
            
            
            for k in output_real:
                prediction_numerical[k].append(
                    outputs[k]
                )
                target_numerical[k].append(
                    labels[k]
                )
                avg_MAE[k].append(losses_step[k])
                    
    for k in predictions_categorical:
            predictions_categorical[k] = torch.cat(predictions_categorical[k])
            target_categorical[k] = torch.cat(target_categorical[k])
               
    for k in prediction_numerical:
        prediction_numerical[k] = torch.cat(prediction_numerical[k])
        target_numerical[k] = torch.cat(target_numerical[k])
   
    
    MAE = {
        k: l1_loss(prediction_numerical[k], target_numerical[k]).item()
        for k in output_real
    }
    
    accuracy = {
            k: multiclass_accuracy(
                predictions_categorical[k],
                target_categorical[k],
                num_classes=output_cat[k],
            )
            for k in output_cat
        }
    
    avg_MAE = {k : sum(avg_MAE[k]) / len(avg_MAE[k]) for k in avg_MAE}
    
    
    Average_total_loss = sum(total_loss) / len(total_loss)
    
    res = {f"{k}_acc" : accuracy[k].item() for k in accuracy} | {f"{k}_mae" : avg_MAE[k] for k in avg_MAE} | {"AVG_total_loss" : Average_total_loss}  | {f"MAE_{k}" : MAE[k] for k in MAE}
    
    print(res)
    
    return res

In [23]:

outputcat = {k : len(list_unique[k]) for k in list_unique}
outputreal = real_value_columns

if ACT_TIME_ONLY:
    outputcat= {"Activity" : outputcat["Activity"]}
    outputreal = ['time:timestamp']

print(outputcat)
print(outputreal)

{'org:resource': 3, 'Activity': 20, 'org:role': 9, 'case:Project': 80, 'case:Task': 598, 'case:OrganizationalEntity': 37, 'case:Activity': 7, 'case:RfpNumber': 6323}
['time:timestamp', 'case:RequestedAmount']


In [24]:
def train_evaluate(config):
    trained_net = train_hgnn(config, output_cat=outputcat, output_real=outputreal, epochs=100)
    return test_hgnn(trained_net, output_cat=outputcat, output_real=outputreal)

In [25]:
import logging

logging.getLogger("root").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
#outputreal = ['time:timestamp']
#outputcat = {"Activity" : outputcat["Activity"]}
#print(outputcat)

In [27]:
best_parameters, values, experiment, model = optimize(
    parameters=[
        {"name": "hid", "type": "choice", "values": [128], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "layers", "type": "choice", "values": [2, 3, 4, 5], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "layers", "type": "choice", "values": [2,4], "value_type": "int", "is_ordered" : True, "sort_values":False},
        {"name": "lr", "type": "range", "bounds": [1e-4, 1e-2], "value_type": "float", "log_scale": True},
        {"name": "batch_size", "type": "choice", "values": [128,256,512], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "batch_size", "type": "choice", "values": [32, 64, 512], "value_type": "int", "is_ordered" : True,"sort_values":False}, 
        {"name": "weight_decay", "type": "range", "bounds" : [1e-2, 1e-1], "value_type": "float", "log_scale" : True}, 
        #{"name": "heads", "type": "choice", "values": [1,2], "value_type": "int", "is_ordered" : True,"sort_values":False},
        #{"name": "heads", "type": "choice", "values": [1], "value_type": "int", "is_ordered" : True,"sort_values":False},
        
        #{"name": "aggregation", "type" : "choice", "values" :["sum", "mean", "max"], "value_type" : "str"}
        {"name": "aggregation", "type" : "choice", "values" :["max","sum"], "value_type" : "str"},
     
    ],
  
    evaluation_function=train_evaluate,
    objective_name='AVG_total_loss',
    arms_per_trial=1,
    minimize = True,
    random_seed = 123,
    total_trials = 50
)

print(best_parameters)
means, covariances = values
print(means)
print(experiment)

[INFO 11-12 10:59:32] ax.service.utils.instantiation: Choice parameter hid contains only one value, converting to a fixed parameter instead.
/home/sebastiano.dissegna/AdaTest/v_env/lib/python3.12/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `is_ordered` is not specified for `ChoiceParameter` "aggregation". Defaulting to `True`  since there are exactly two choices.. To override this behavior (or avoid this warning), specify `is_ordered` during `ChoiceParameter` construction. Note that choice parameters with exactly 2 choices are always considered ordered and that the user-supplied `is_ordered` has no effect in this particular case.
  return ChoiceParameter(
/home/sebastiano.dissegna/AdaTest/v_env/lib/python3.12/site-packages/ax/service/utils/instantiation.py:248: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "aggregation". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warnin

{'layers': 2, 'lr': 0.0014528509914685322, 'batch_size': 256, 'weight_decay': 0.09787123410578823, 'aggregation': 'sum', 'hid': 128}


  0%|          | 0/100 [00:00<?, ?it/s]

[INFO 11-12 11:04:42] ax.core.experiment: Attached data has some metrics ({'MAE_time:timestamp', 'case:RequestedAmount_mae', 'case:RfpNumber_acc', 'Activity_acc', 'case:Task_acc', 'org:role_acc', 'case:OrganizationalEntity_acc', 'case:Activity_acc', 'time:timestamp_mae', 'MAE_case:RequestedAmount', 'org:resource_acc', 'case:Project_acc'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[INFO 11-12 11:04:42] ax.service.managed_loop: Running optimization trial 2...
[ERROR 11-12 11:04:42] ax.core.observation: Data contains metric Activity_acc that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics

{'org:resource_acc': 0.931147038936615, 'Activity_acc': 0.8233917951583862, 'org:role_acc': 0.8227332830429077, 'case:Project_acc': 0.2770439088344574, 'case:Task_acc': 0.7536268830299377, 'case:OrganizationalEntity_acc': 0.22061970829963684, 'case:Activity_acc': 0.9820506572723389, 'case:RfpNumber_acc': 0.03960190713405609, 'time:timestamp_mae': 0.006602124336899983, 'case:RequestedAmount_mae': 0.25238548081230233, 'AVG_total_loss': 17.532494372507145, 'MAE_time:timestamp': 0.006709863431751728, 'MAE_case:RequestedAmount': 0.2524721920490265}
{'layers': 4, 'lr': 0.00029413132884013236, 'batch_size': 512, 'weight_decay': 0.020455714056727532, 'aggregation': 'max', 'hid': 128}


...

/home/sebastiano.dissegna/AdaTest/v_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


{'layers': 4, 'lr': 0.01, 'batch_size': 128, 'weight_decay': 0.019480125717349497, 'aggregation': 'sum', 'hid': 128}


  0%|          | 0/100 [00:00<?, ?it/s]

[INFO 11-12 15:25:22] ax.core.experiment: Attached data has some metrics ({'MAE_time:timestamp', 'case:RequestedAmount_mae', 'case:RfpNumber_acc', 'Activity_acc', 'case:Task_acc', 'org:role_acc', 'case:OrganizationalEntity_acc', 'case:Activity_acc', 'time:timestamp_mae', 'MAE_case:RequestedAmount', 'org:resource_acc', 'case:Project_acc'}) that are not among the metrics on this experiment. Note that attaching data will not automatically add those metrics to the experiment. For these metrics to be automatically fetched by `experiment.fetch_data`, add them via `experiment.add_tracking_metric` or update the experiment's optimization config.
[ERROR 11-12 15:25:22] ax.core.observation: Data contains metric Activity_acc that has not been added to the experiment. You can either update the `optimization_config` or attach it as a tracking metric using `Experiment.add_tracking_metrics` or `AxClient.add_tracking_metrics`. Ignoring all data for metric Activity_acc.
NoneType: None
[ERROR 11-12 15:25

{'org:resource_acc': 0.9789360761642456, 'Activity_acc': 0.9444190263748169, 'org:role_acc': 0.9635697603225708, 'case:Project_acc': 0.9082470536231995, 'case:Task_acc': 0.7536268830299377, 'case:OrganizationalEntity_acc': 0.9010804295539856, 'case:Activity_acc': 0.9912554621696472, 'case:RfpNumber_acc': 0.03960190713405609, 'time:timestamp_mae': 0.006366417458694842, 'case:RequestedAmount_mae': 0.20591616327012027, 'AVG_total_loss': 11.88947294083536, 'MAE_time:timestamp': 0.006485761143267155, 'MAE_case:RequestedAmount': 0.2069842666387558}
{'layers': 4, 'lr': 0.01, 'batch_size': 128, 'weight_decay': 0.0172933458234693, 'aggregation': 'sum', 'hid': 128}
{'AVG_total_loss': 11.951471678514562}
Experiment(None)


In [28]:
from ax.service.utils.report_utils import exp_to_df

results = exp_to_df(experiment)

[WARNING 11-12 15:25:23] ax.service.utils.report_utils: Column reason missing for all trials. Not appending column.


In [29]:
results.sort_values(by="AVG_total_loss")

,trial_index,arm_name,trial_status,generation_method,AVG_total_loss,Activity_acc,MAE_case:RequestedAmount,MAE_time:timestamp,case:Activity_acc,case:OrganizationalEntity_acc,case:Project_acc,case:RequestedAmount_mae,case:RfpNumber_acc,case:Task_acc,org:resource_acc,org:role_acc,time:timestamp_mae,layers,lr,batch_size,weight_decay,aggregation,hid
46,46,46_0,COMPLETED,BoTorch,11.735283,0.940351,0.127704,0.006110,0.990335,0.915146,0.884735,0.126398,0.039602,0.754291,0.971439,0.967437,0.005992,4,0.010000,128,0.019074,sum,128
34,34,34_0,COMPLETED,BoTorch,11.839662,0.944724,0.125377,0.006458,0.992687,0.916064,0.901515,0.124329,0.039602,0.754444,0.980211,0.957820,0.006341,4,0.010000,128,0.022975,sum,128
40,40,40_0,COMPLETED,BoTorch,11.861984,0.937046,0.185284,0.007684,0.991051,0.911987,0.896364,0.184269,0.039602,0.757152,0.973020,0.960313,0.007583,4,0.010000,128,0.022416,sum,128
39,39,39_0,COMPLETED,BoTorch,11.878641,0.938012,0.156870,0.006835,0.987062,0.909031,0.881369,0.156214,0.039602,0.754444,0.976641,0.965809,0.006736,4,0.010000,128,0.022393,sum,128
49,49,49_0,COMPLETED,BoTorch,11.889473,0.944419,0.206984,0.006486,0.991255,0.901080,0.908247,0.205916,0.039602,0.753627,0.978936,0.963570,0.006366,4,0.010000,128,0.019480,sum,128
43,43,43_0,COMPLETED,BoTorch,11.917010,0.934350,0.175226,0.007012,0.993148,0.920497,0.889937,0.174374,0.039602,0.753627,0.977253,0.961585,0.006900,4,0.010000,128,0.019647,sum,128
11,11,11_0,COMPLETED,BoTorch,11.926921,0.947114,0.139186,0.006472,0.993761,0.912292,0.904932,0.138116,0.039602,0.753627,0.974805,0.956548,0.006353,4,0.010000,128,0.019508,sum,128
45,45,45_0,COMPLETED,BoTorch,11.937763,0.917671,0.175542,0.006503,0.990233,0.887269,0.900342,0.174891,0.039602,0.758684,0.984291,0.965758,0.006395,4,0.010000,128,0.019062,sum,128
44,44,44_0,COMPLETED,BoTorch,11.981890,0.928502,0.175479,0.006552,0.988289,0.903578,0.884327,0.174436,0.039291,0.756130,0.981792,0.953648,0.006435,4,0.010000,128,0.019592,sum,128
19,19,19_0,COMPLETED,BoTorch,11.990137,0.926112,0.195609,0.007245,0.992892,0.870808,0.899169,0.194220,0.039602,0.755466,0.980466,0.957769,0.007163,4,0.010000,128,0.019862,sum,128


In [30]:
results = results.sort_values(by="AVG_total_loss")

In [31]:
results.to_csv(f"results/results_CAISE/{dataset}_CONFIGS.csv", sep=",")

## Test on different test sets

In [32]:
test_types = ["even", "odd", "window", "random", "attr_level"]

In [33]:
import pandas


def create_df(results):
    res = {}
    
    for k in results[0]:
        res[k] = [x[k] for x in results]
    
    res = pandas.DataFrame(data=res)
    
    return res, res.mean(), res.std()
        
    

In [34]:
def test_multi(config, outputcat, outputreal, num_runs=10, num_epochs=20):
    
    res = {}
    
    save_path = f"results/results_CAISE/{dataset}/"
    
    if not os.path.isdir(save_path):
        os.makedirs(save_path)
    
    for i in range(num_runs):
        
        print(f"Run {i}")
        
        net = train_hgnn(
                config, 
                outputcat,
                outputreal,
                num_epochs
            )
        
        
        
        for test_type in test_types:
            
            print(f"Test type {test_type}")
            
            with open(data_dir_graphs + dataset + f"_TEST_V2_repair_{test_type}.pkl", "rb") as f:
                X = pickle.load(f)    
                
            for i in range(len(X)):
                X[i] = transform(X[i])

            if test_type not in res:
                res[test_type] = []
            
           
                
            
                
            res[test_type].append(
                test_hgnn(
                    net,
                    outputcat,
                    outputreal,
                    test_graphs=X
                )
            )
    
    for test_type in test_types:    
        results_table, means, stds = create_df(res[test_type])
                
        results_table.to_csv(f"{save_path}{test_type}_V2_RESULTS_END.csv", sep=",", index=False)
        
        pd.DataFrame(data={"mean" : means, "std" : stds}).to_csv(f"{save_path}{test_type}_V2_MEAN_STD_END.csv", sep=",")
        
        print(test_type)
        print(pd.DataFrame(data={"mean" : means, "std" : stds}))
        
        
    
    return res

In [ ]:
dataset

In [ ]:
best_parameters

In [35]:
res = test_multi(best_parameters, outputcat, outputreal, 10, 100)

Run 0
{'layers': 4, 'lr': 0.01, 'batch_size': 128, 'weight_decay': 0.0172933458234693, 'aggregation': 'sum', 'hid': 128}


  0%|          | 0/100 [00:00<?, ?it/s]

Test type even
{'org:resource_acc': 1.0, 'Activity_acc': 0.9672920107841492, 'org:role_acc': 0.9918230175971985, 'case:Project_acc': 0.9367484450340271, 'case:Task_acc': 0.7561327815055847, 'case:OrganizationalEntity_acc': 0.9514189958572388, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.044733043760061264, 'time:timestamp_mae': 0.00896613435311751, 'case:RequestedAmount_mae': 0.04669454896991903, 'AVG_total_loss': 11.46605064959096, 'MAE_time:timestamp': 0.008677461184561253, 'MAE_case:RequestedAmount': 0.04690558463335037}
Test type odd
{'org:resource_acc': 0.9993954300880432, 'Activity_acc': 0.9637243151664734, 'org:role_acc': 0.9924426078796387, 'case:Project_acc': 0.9319831132888794, 'case:Task_acc': 0.7551390528678894, 'case:OrganizationalEntity_acc': 0.9489117860794067, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.03174123540520668, 'time:timestamp_mae': 0.009411917195062746, 'case:RequestedAmount_mae': 0.06112187321890484, 'AVG_total_loss': 11.800626447985202, 'MAE_tim

  0%|          | 0/100 [00:00<?, ?it/s]

Test type even
{'org:resource_acc': 0.9995189905166626, 'Activity_acc': 0.9694564938545227, 'org:role_acc': 0.9920635223388672, 'case:Project_acc': 0.941077470779419, 'case:Task_acc': 0.7570948004722595, 'case:OrganizationalEntity_acc': 0.9540644884109497, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.044733043760061264, 'time:timestamp_mae': 0.005737222053787925, 'case:RequestedAmount_mae': 0.10044380277395248, 'AVG_total_loss': 11.15396140712652, 'MAE_time:timestamp': 0.005589568056166172, 'MAE_case:RequestedAmount': 0.10030018538236618}
Test type odd
{'org:resource_acc': 0.9993954300880432, 'Activity_acc': 0.9866989254951477, 'org:role_acc': 0.9927449226379395, 'case:Project_acc': 0.9365175366401672, 'case:Task_acc': 0.7533252835273743, 'case:OrganizationalEntity_acc': 0.9489117860794067, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.03174123540520668, 'time:timestamp_mae': 0.006262020800601353, 'case:RequestedAmount_mae': 0.08987421610138634, 'AVG_total_loss': 11.3567400332

...

Test type even
{'org:resource_acc': 0.9997594952583313, 'Activity_acc': 0.9684944748878479, 'org:role_acc': 0.9947090148925781, 'case:Project_acc': 0.9405964612960815, 'case:Task_acc': 0.7554112672805786, 'case:OrganizationalEntity_acc': 0.9160654544830322, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.044733043760061264, 'time:timestamp_mae': 0.005728889574212107, 'case:RequestedAmount_mae': 0.055020289664918724, 'AVG_total_loss': 11.252926186065782, 'MAE_time:timestamp': 0.005483168642967939, 'MAE_case:RequestedAmount': 0.05493857339024544}
Test type odd
{'org:resource_acc': 0.9993954300880432, 'Activity_acc': 0.9631197452545166, 'org:role_acc': 0.990024209022522, 'case:Project_acc': 0.9383313655853271, 'case:Task_acc': 0.7512091994285583, 'case:OrganizationalEntity_acc': 0.9105199575424194, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.03174123540520668, 'time:timestamp_mae': 0.007462519094009291, 'case:RequestedAmount_mae': 0.056019772521474144, 'AVG_total_loss': 11.5498357

  0%|          | 0/100 [00:00<?, ?it/s]

Test type even
{'org:resource_acc': 0.9995189905166626, 'Activity_acc': 0.9684944748878479, 'org:role_acc': 0.9915825128555298, 'case:Project_acc': 0.945406436920166, 'case:Task_acc': 0.7568542957305908, 'case:OrganizationalEntity_acc': 0.9526214599609375, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.044733043760061264, 'time:timestamp_mae': 0.0073779931884597645, 'case:RequestedAmount_mae': 0.05109399049119516, 'AVG_total_loss': 10.95847941759381, 'MAE_time:timestamp': 0.007088411133736372, 'MAE_case:RequestedAmount': 0.05118352919816971}
Test type odd
{'org:resource_acc': 0.9993954300880432, 'Activity_acc': 0.9637243151664734, 'org:role_acc': 0.9918379783630371, 'case:Project_acc': 0.9401451349258423, 'case:Task_acc': 0.755743682384491, 'case:OrganizationalEntity_acc': 0.9498186707496643, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.03174123540520668, 'time:timestamp_mae': 0.00858039887283336, 'case:RequestedAmount_mae': 0.04152404991063205, 'AVG_total_loss': 11.21176506974

  0%|          | 0/100 [00:00<?, ?it/s]

Test type even
{'org:resource_acc': 1.0, 'Activity_acc': 0.969215989112854, 'org:role_acc': 0.9927850365638733, 'case:Project_acc': 0.93241947889328, 'case:Task_acc': 0.7539682388305664, 'case:OrganizationalEntity_acc': 0.947811484336853, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.044733043760061264, 'time:timestamp_mae': 0.007339341311969541, 'case:RequestedAmount_mae': 0.10156764631921594, 'AVG_total_loss': 11.283414655111052, 'MAE_time:timestamp': 0.007141017355024815, 'MAE_case:RequestedAmount': 0.1016201302409172}
Test type odd
{'org:resource_acc': 0.9993954300880432, 'Activity_acc': 0.9591898918151855, 'org:role_acc': 0.9897218942642212, 'case:Project_acc': 0.9286578297615051, 'case:Task_acc': 0.75, 'case:OrganizationalEntity_acc': 0.9431681036949158, 'case:Activity_acc': 1.0, 'case:RfpNumber_acc': 0.03174123540520668, 'time:timestamp_mae': 0.006847506346689029, 'case:RequestedAmount_mae': 0.05946036699143323, 'AVG_total_loss': 11.554417139474852, 'MAE_time:timestamp': 0.0